## LightGBM

In [44]:
import pandas as pd
from sklearn.model_selection import train_test_split
#from lightgbm import LGBMClassifier
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

In [37]:
train_df = pd.read_csv("../Dataset/train.csv")
train_df = train_df.drop(columns=['id'])
test_df = pd.read_csv("../Dataset/test.csv")
test_df = test_df.drop(columns=['id'])

In [27]:
# Step 2 - separate features and target
X = train_df.drop('Heart Disease', axis=1)
y = train_df['Heart Disease']

In [28]:
# Step 3- converting target to binary
y = y.map({'Presence': 1, 'Absence': 0})

In [29]:
# Step 4, identifying feature types
numerical_features = ['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression']
categorical_features = [col for col in X.columns if col not in numerical_features]

In [38]:
# steo 5 converting categorical features to category dtype
for col in categorical_features:
    X[col] = X[col].astype('category')
    test_df[col] = test_df[col].astype('category')

In [39]:
# Step 6, Train, Test Split (stratify to maintain class distribution)
X_train, X_validation, y_train, y_validation = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [32]:
# Step 7 - Minimal Preprocessing (All preprocessing must be fit ONLY on the trainin portion, not validation).
# Does NOT need scaling
# Does NOT need normalization
# Handles skewed distributions well

In [45]:
# Step 8, building LightGBM model
model = lgb.LGBMClassifier(
    objective="binary",
    metric="auc",
    learning_rate=0.05,
    num_leaves=64,
    max_depth=-1,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    min_data_in_leaf=100,
    random_state=42,
    n_estimators=1000  # large number, early stopping will handle
)

In [ ]:
# Step 9 - fit model on training data
model.fit(
    X_train, y_train, 
    eval_set=[(X_validation, y_validation)], 
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=10),
        lgb.log_evaluation(100)
    ]
)

In [48]:
# Step 10 - evaluation of the base model (LightGBM with single-train-test-split) on validation set using ROC-AUC
y_valid_pred = model.predict_proba(X_validation)[:, 1]
auc_score = roc_auc_score(y_validation, y_valid_pred)
print(f"Validation ROC-AUC on VALIDATION_SET: {auc_score:.5f}")

[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
Validation ROC-AUC on VALIDATION_SET: 0.95607


In [ ]:
train_df.head(3)

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence


In [36]:
test_df.head(3)

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium
0,630000,58,1,3,120,288,0,2,145,1,0.8,2,3,3
1,630001,55,0,2,120,209,0,0,172,0,0.0,1,0,3
2,630002,54,1,4,120,268,0,0,150,1,0.0,2,3,7
